# 第11章: 慣性オドメトリ - IMUプリインテグレーション

**SLAM Handbook Chapter 11, Section 11.1-11.4**

このノートブックでは、IMU（慣性計測ユニット）のモデルとプリインテグレーションの基礎を学びます。

## 学習内容
1. IMUセンサーモデル（ジャイロスコープ + 加速度計、バイアス）
2. オイラー積分によるIMU積分
3. IMUプリインテグレーション（Eq 11.14: $\Delta R$, $\Delta v$, $\Delta p$）
4. ファクターグラフにおけるプリインテグレーションファクター
5. ノイズ伝播のデモ

IMUはカメラやLiDARと融合することで、高頻度かつ堅牢な状態推定を実現します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 11.1 IMUセンサーモデル

**SLAM Handbook Section 11.1**

IMUはジャイロスコープ（角速度）と加速度計（加速度）から構成されます。

### ジャイロスコープモデル
$$\tilde{\boldsymbol{\omega}} = \boldsymbol{\omega} + \mathbf{b}_g + \mathbf{n}_g$$

### 加速度計モデル
$$\tilde{\mathbf{a}} = \mathbf{R}^T(\mathbf{a} - \mathbf{g}) + \mathbf{b}_a + \mathbf{n}_a$$

ここで：
- $\mathbf{b}_g, \mathbf{b}_a$: バイアス（ランダムウォーク: $\dot{\mathbf{b}} = \mathbf{n}_b$）
- $\mathbf{n}_g, \mathbf{n}_a$: 白色ガウスノイズ
- $\mathbf{g} = [0, 0, -9.81]^T$: 重力ベクトル
- $\mathbf{R}$: ワールド座標からボディ座標への回転行列

In [ ]:
def skew(v):
    """Create skew-symmetric matrix from 3D vector."""
    return np.array([[0, -v[2], v[1]],
                     [v[2], 0, -v[0]],
                     [-v[1], v[0], 0]])

def exp_so3(omega):
    """Exponential map SO(3): rotation vector -> rotation matrix (Rodrigues)."""
    theta = np.linalg.norm(omega)
    if theta < 1e-10:
        return np.eye(3)
    k = omega / theta
    K = skew(k)
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * K @ K

class IMUSimulator:
    """Simulate IMU measurements with noise and bias."""

    def __init__(self, gyro_noise=0.01, accel_noise=0.1,
                 gyro_bias_noise=0.001, accel_bias_noise=0.01):
        self.gyro_noise = gyro_noise      # rad/s/sqrt(Hz)
        self.accel_noise = accel_noise     # m/s^2/sqrt(Hz)
        self.gyro_bias_noise = gyro_bias_noise
        self.accel_bias_noise = accel_bias_noise
        self.gravity = np.array([0, 0, -9.81])

    def generate_trajectory(self, dt=0.01, duration=10.0):
        """Generate a circular trajectory with known ground truth."""
        n_steps = int(duration / dt)
        t = np.arange(n_steps) * dt

        # Circular motion in XY plane with oscillation in Z
        radius = 5.0
        omega_circle = 0.5  # rad/s

        # Ground truth states
        positions = np.zeros((n_steps, 3))
        velocities = np.zeros((n_steps, 3))
        rotations = np.zeros((n_steps, 3, 3))
        true_omega = np.zeros((n_steps, 3))
        true_accel = np.zeros((n_steps, 3))

        for i in range(n_steps):
            angle = omega_circle * t[i]
            # Position
            positions[i] = [radius * np.cos(angle), radius * np.sin(angle),
                           0.5 * np.sin(2 * angle)]
            # Velocity
            velocities[i] = [-radius * omega_circle * np.sin(angle),
                             radius * omega_circle * np.cos(angle),
                             1.0 * omega_circle * np.cos(2 * angle)]
            # Rotation: body z-axis up, x-axis along velocity
            vn = velocities[i] / (np.linalg.norm(velocities[i]) + 1e-10)
            z_axis = np.array([0, 0, 1.0])
            y_axis = np.cross(z_axis, vn)
            y_axis /= (np.linalg.norm(y_axis) + 1e-10)
            x_axis = np.cross(y_axis, z_axis)
            rotations[i] = np.column_stack([x_axis, y_axis, z_axis])

            # True angular velocity (body frame)
            true_omega[i] = [0, 0, omega_circle]
            # True acceleration (world frame)
            true_accel[i] = [-radius * omega_circle**2 * np.cos(angle),
                            -radius * omega_circle**2 * np.sin(angle),
                            -2.0 * omega_circle**2 * np.sin(2 * angle)]

        return t, positions, velocities, rotations, true_omega, true_accel

    def generate_measurements(self, t, rotations, true_omega, true_accel):
        """Generate noisy IMU measurements."""
        n_steps = len(t)
        dt = t[1] - t[0]

        gyro_meas = np.zeros((n_steps, 3))
        accel_meas = np.zeros((n_steps, 3))
        bg = np.zeros(3)  # Gyro bias (evolving)
        ba = np.array([0.05, -0.03, 0.02])  # Accel bias (evolving)

        for i in range(n_steps):
            R = rotations[i]
            # Gyro: body-frame angular velocity + bias + noise
            gyro_meas[i] = true_omega[i] + bg + \
                np.random.normal(0, self.gyro_noise / np.sqrt(dt), 3)
            # Accel: R^T(a - g) + bias + noise
            accel_meas[i] = R.T @ (true_accel[i] - self.gravity) + ba + \
                np.random.normal(0, self.accel_noise / np.sqrt(dt), 3)
            # Random walk bias
            bg += np.random.normal(0, self.gyro_bias_noise * np.sqrt(dt), 3)
            ba += np.random.normal(0, self.accel_bias_noise * np.sqrt(dt), 3)

        return gyro_meas, accel_meas

# Generate trajectory and IMU data
imu = IMUSimulator()
t, pos_true, vel_true, rot_true, omega_true, accel_true = imu.generate_trajectory()
gyro_meas, accel_meas = imu.generate_measurements(t, rot_true, omega_true, accel_true)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trajectory
ax = axes[0, 0]
ax.plot(pos_true[:, 0], pos_true[:, 1], 'b-', linewidth=2)
ax.plot(pos_true[0, 0], pos_true[0, 1], 'go', markersize=10, label='Start')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Ground Truth Trajectory (XY)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()

# Gyro measurements
ax = axes[0, 1]
ax.plot(t, gyro_meas[:, 0], alpha=0.7, label='$\\omega_x$')
ax.plot(t, gyro_meas[:, 1], alpha=0.7, label='$\\omega_y$')
ax.plot(t, gyro_meas[:, 2], alpha=0.7, label='$\\omega_z$')
ax.axhline(y=0.5, color='k', linestyle='--', alpha=0.3, label='True $\\omega_z$')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Angular velocity [rad/s]')
ax.set_title('Gyroscope Measurements')
ax.legend()
ax.grid(True, alpha=0.3)

# Accel measurements
ax = axes[1, 0]
ax.plot(t, accel_meas[:, 0], alpha=0.7, label='$a_x$')
ax.plot(t, accel_meas[:, 1], alpha=0.7, label='$a_y$')
ax.plot(t, accel_meas[:, 2], alpha=0.7, label='$a_z$')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Acceleration [m/s²]')
ax.set_title('Accelerometer Measurements')
ax.legend()
ax.grid(True, alpha=0.3)

# 3D trajectory
ax = fig.add_subplot(2, 2, 4, projection='3d')
ax.plot(pos_true[:, 0], pos_true[:, 1], pos_true[:, 2], 'b-', linewidth=2)
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('z [m]')
ax.set_title('3D Trajectory')

plt.tight_layout()
plt.show()

## 11.2 オイラー積分とIMUプリインテグレーション

**SLAM Handbook Section 11.2-11.3, Eq 11.14**

### 従来のIMU積分（グローバル座標）
$$\mathbf{R}_{k+1} = \mathbf{R}_k \cdot \text{Exp}((\tilde{\boldsymbol{\omega}}_k - \mathbf{b}_g) \Delta t)$$
$$\mathbf{v}_{k+1} = \mathbf{v}_k + \mathbf{g}\Delta t + \mathbf{R}_k(\tilde{\mathbf{a}}_k - \mathbf{b}_a)\Delta t$$
$$\mathbf{p}_{k+1} = \mathbf{p}_k + \mathbf{v}_k\Delta t + \frac{1}{2}\mathbf{g}\Delta t^2 + \frac{1}{2}\mathbf{R}_k(\tilde{\mathbf{a}}_k - \mathbf{b}_a)\Delta t^2$$

### プリインテグレーション量（ローカル座標、初期状態非依存）
$$\Delta \mathbf{R}_{ij} = \prod_{k=i}^{j-1} \text{Exp}((\tilde{\boldsymbol{\omega}}_k - \mathbf{b}_g)\Delta t)$$
$$\Delta \mathbf{v}_{ij} = \sum_{k=i}^{j-1} \Delta \mathbf{R}_{ik}(\tilde{\mathbf{a}}_k - \mathbf{b}_a)\Delta t$$
$$\Delta \mathbf{p}_{ij} = \sum_{k=i}^{j-1} \Delta \mathbf{v}_{ik}\Delta t + \frac{1}{2}\Delta \mathbf{R}_{ik}(\tilde{\mathbf{a}}_k - \mathbf{b}_a)\Delta t^2$$

プリインテグレーションの利点：初期状態（$\mathbf{R}_i, \mathbf{v}_i, \mathbf{p}_i$）が変更されても再積分が不要。

In [ ]:
def imu_euler_integration(gyro_meas, accel_meas, dt, R0, v0, p0, bg, ba, gravity):
    """Standard Euler integration of IMU measurements in world frame."""
    n = len(gyro_meas)
    R = R0.copy()
    v = v0.copy()
    p = p0.copy()

    positions = [p.copy()]
    velocities = [v.copy()]

    for k in range(n):
        omega = gyro_meas[k] - bg
        acc = accel_meas[k] - ba

        # Update rotation
        R = R @ exp_so3(omega * dt)
        # Update velocity
        v = v + gravity * dt + R @ acc * dt
        # Update position
        p = p + v * dt + 0.5 * gravity * dt**2 + 0.5 * R @ acc * dt**2

        positions.append(p.copy())
        velocities.append(v.copy())

    return np.array(positions), np.array(velocities)

class IMUPreintegration:
    """IMU preintegration on manifold (Eq 11.14)."""

    def __init__(self, bg=np.zeros(3), ba=np.zeros(3)):
        self.bg = bg.copy()
        self.ba = ba.copy()
        self.delta_R = np.eye(3)
        self.delta_v = np.zeros(3)
        self.delta_p = np.zeros(3)
        self.dt_sum = 0.0
        # Noise covariance (for noise propagation)
        self.covariance = np.zeros((9, 9))  # [dR, dv, dp]

    def integrate(self, gyro, accel, dt, gyro_noise_cov=None, accel_noise_cov=None):
        """Integrate one IMU measurement."""
        omega = gyro - self.bg
        acc = accel - self.ba

        # Preintegration update (Eq 11.14)
        dR = exp_so3(omega * dt)

        # Update delta_p first (uses current delta_v and delta_R)
        self.delta_p += self.delta_v * dt + 0.5 * self.delta_R @ acc * dt**2
        # Update delta_v
        self.delta_v += self.delta_R @ acc * dt
        # Update delta_R
        self.delta_R = self.delta_R @ dR

        self.dt_sum += dt

        # Noise propagation (linearized)
        if gyro_noise_cov is not None and accel_noise_cov is not None:
            A = np.zeros((9, 9))
            B = np.zeros((9, 6))
            # Rotation part
            A[0:3, 0:3] = dR.T
            # Velocity part
            A[3:6, 0:3] = -self.delta_R @ skew(acc) * dt
            A[3:6, 3:6] = np.eye(3)
            # Position part
            A[6:9, 0:3] = -0.5 * self.delta_R @ skew(acc) * dt**2
            A[6:9, 3:6] = np.eye(3) * dt
            A[6:9, 6:9] = np.eye(3)

            B[0:3, 0:3] = np.eye(3) * dt  # dR vs gyro noise
            B[3:6, 3:6] = self.delta_R * dt  # dv vs accel noise
            B[6:9, 3:6] = 0.5 * self.delta_R * dt**2  # dp vs accel noise

            Q = np.zeros((6, 6))
            Q[0:3, 0:3] = gyro_noise_cov
            Q[3:6, 3:6] = accel_noise_cov

            self.covariance = A @ self.covariance @ A.T + B @ Q @ B.T

    def predict(self, R_i, v_i, p_i, gravity):
        """Predict state at time j from state at time i."""
        dt = self.dt_sum
        R_j = R_i @ self.delta_R
        v_j = v_i + gravity * dt + R_i @ self.delta_v
        p_j = p_i + v_i * dt + 0.5 * gravity * dt**2 + R_i @ self.delta_p
        return R_j, v_j, p_j

# Compare standard integration vs preintegration
dt = 0.01
gravity = np.array([0, 0, -9.81])
bg_est = np.zeros(3)  # Estimated biases (imperfect)
ba_est = np.array([0.05, -0.03, 0.02])

# Standard integration
pos_integrated, vel_integrated = imu_euler_integration(
    gyro_meas, accel_meas, dt, rot_true[0], vel_true[0], pos_true[0],
    bg_est, ba_est, gravity)

# Preintegration over windows
window_size = 100  # 1 second windows
n_windows = len(gyro_meas) // window_size
preint_positions = [pos_true[0].copy()]

for w in range(n_windows):
    start = w * window_size
    end = (w + 1) * window_size

    preint = IMUPreintegration(bg=bg_est, ba=ba_est)
    gyro_cov = np.eye(3) * imu.gyro_noise**2
    accel_cov = np.eye(3) * imu.accel_noise**2

    for k in range(start, end):
        preint.integrate(gyro_meas[k], accel_meas[k], dt, gyro_cov, accel_cov)

    # Use ground truth at window start for prediction
    R_pred, v_pred, p_pred = preint.predict(
        rot_true[start], vel_true[start], pos_true[start], gravity)
    preint_positions.append(p_pred.copy())

preint_positions = np.array(preint_positions)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(pos_true[:, 0], pos_true[:, 1], 'b-', linewidth=2, label='Ground Truth')
ax.plot(pos_integrated[:, 0], pos_integrated[:, 1], 'r--', alpha=0.7, label='Euler Integration')
ax.plot(preint_positions[:, 0], preint_positions[:, 1], 'g*-', markersize=8,
        label='Preintegration (1s windows)')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Trajectory Comparison (XY)')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

# Position error over time
ax = axes[1]
euler_error = np.linalg.norm(pos_integrated[:-1] - pos_true, axis=1)
ax.plot(t, euler_error, 'r-', alpha=0.7, label='Euler Integration Error')
preint_times = np.arange(n_windows + 1) * window_size * dt
preint_error = np.linalg.norm(preint_positions - pos_true[::window_size][:n_windows+1], axis=1)
ax.plot(preint_times[:len(preint_error)], preint_error, 'g*-', markersize=8,
        label='Preintegration Error (per window)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Position Error [m]')
ax.set_title('Position Error Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Euler integration error: {euler_error[-1]:.3f} m")
print(f"Mean preintegration window error: {preint_error.mean():.3f} m")

## 11.3 ファクターグラフにおけるプリインテグレーションファクター

**SLAM Handbook Section 11.4**

プリインテグレーションファクターは、2つのキーフレーム $i, j$ 間のIMU制約をファクターグラフに組み込みます。

残差の定義：
$$\mathbf{r}_{\Delta R} = \text{Log}(\Delta\tilde{\mathbf{R}}_{ij}^T \mathbf{R}_i^T \mathbf{R}_j)$$
$$\mathbf{r}_{\Delta v} = \mathbf{R}_i^T(\mathbf{v}_j - \mathbf{v}_i - \mathbf{g}\Delta t_{ij}) - \Delta\tilde{\mathbf{v}}_{ij}$$
$$\mathbf{r}_{\Delta p} = \mathbf{R}_i^T(\mathbf{p}_j - \mathbf{p}_i - \mathbf{v}_i\Delta t_{ij} - \frac{1}{2}\mathbf{g}\Delta t_{ij}^2) - \Delta\tilde{\mathbf{p}}_{ij}$$

以下では簡単なファクターグラフを構成して可視化します。

In [ ]:
# Visualize factor graph with preintegration factors and noise covariance
from matplotlib.patches import Ellipse

# Compute preintegration for each window and collect covariances
preint_list = []
for w in range(n_windows):
    start = w * window_size
    end = (w + 1) * window_size

    preint = IMUPreintegration(bg=bg_est, ba=ba_est)
    gyro_cov = np.eye(3) * imu.gyro_noise**2
    accel_cov = np.eye(3) * imu.accel_noise**2

    for k in range(start, end):
        preint.integrate(gyro_meas[k], accel_meas[k], dt, gyro_cov, accel_cov)
    preint_list.append(preint)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Factor graph visualization
ax = axes[0]
kf_positions = pos_true[::window_size][:n_windows + 1]

# Draw IMU preintegration factors (edges)
for w in range(n_windows):
    ax.annotate('', xy=kf_positions[w + 1, :2], xytext=kf_positions[w, :2],
                arrowprops=dict(arrowstyle='->', color='red', lw=2))

# Draw keyframe nodes
for i, p in enumerate(kf_positions):
    ax.plot(p[0], p[1], 'ko', markersize=12, zorder=5)
    ax.plot(p[0], p[1], 'wo', markersize=8, zorder=6)
    ax.text(p[0] + 0.3, p[1] + 0.3, f'$x_{i}$', fontsize=10, zorder=7)

# Prior factor on first pose
ax.plot(kf_positions[0, 0], kf_positions[0, 1], 'gs', markersize=15, zorder=4,
        label='Prior factor')

# Legend entries
ax.plot([], [], 'r-', linewidth=2, label='IMU preintegration factor')
ax.plot([], [], 'ko', markersize=10, label='Keyframe (pose, vel, bias)')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Factor Graph with IMU Preintegration')
ax.set_aspect('equal')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Noise propagation: covariance growth over time
ax = axes[1]
position_sigmas = []
for preint in preint_list:
    # Extract position covariance (last 3x3 block)
    pos_cov = preint.covariance[6:9, 6:9]
    sigma = np.sqrt(np.trace(pos_cov))
    position_sigmas.append(sigma)

rotation_sigmas = []
for preint in preint_list:
    rot_cov = preint.covariance[0:3, 0:3]
    sigma = np.sqrt(np.trace(rot_cov))
    rotation_sigmas.append(sigma)

velocity_sigmas = []
for preint in preint_list:
    vel_cov = preint.covariance[3:6, 3:6]
    sigma = np.sqrt(np.trace(vel_cov))
    velocity_sigmas.append(sigma)

window_indices = np.arange(1, n_windows + 1)
ax.bar(window_indices - 0.2, rotation_sigmas, 0.2, label='Rotation $\\sigma$', alpha=0.7)
ax.bar(window_indices, velocity_sigmas, 0.2, label='Velocity $\\sigma$', alpha=0.7)
ax.bar(window_indices + 0.2, position_sigmas, 0.2, label='Position $\\sigma$', alpha=0.7)
ax.set_xlabel('Window index')
ax.set_ylabel('Standard deviation')
ax.set_title('Preintegration Noise Propagation\n(per 1-second window)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 演習問題

1. **バイアス推定**: バイアスの初期推定値を意図的に間違え（例: `bg_est = [0.1, 0, 0]`）、積分精度への影響を観察せよ。
2. **積分ウィンドウサイズ**: `window_size` を50, 100, 200, 500と変化させたとき、プリインテグレーションの共分散がどう変化するか調べよ。
3. **ミッドポイント積分**: オイラー法の代わりにミッドポイント法（台形則）を実装し、精度の改善を確認せよ。

## まとめ

本ノートブックでは以下を学びました：

1. **IMUセンサーモデル**: ジャイロスコープと加速度計のノイズ・バイアスモデル
2. **オイラー積分**: ワールド座標系でのIMU状態積分と、長時間でのドリフト累積
3. **プリインテグレーション**: 初期状態非依存な相対運動量（$\Delta R, \Delta v, \Delta p$）の計算（Eq 11.14）
4. **ファクターグラフ統合**: プリインテグレーションファクターによるキーフレーム間制約
5. **ノイズ伝播**: プリインテグレーション中の共分散の離散時間伝播

IMUプリインテグレーションは、VIO（Visual-Inertial Odometry）やLIO（LiDAR-Inertial Odometry）の核となる技術です。

**参考**: SLAM Handbook Chapter 11